# 原ER运行时ZIP检查与原断点恢复

**复制下一格到原ER Notebook中，等⑤结束后运行。不要在正在计算时换源码。** 首次RESUME=False只检查，原文件保留。改True才启动恢复及原⑤，已有成功账户跳过。若本地ZIP精确匹配冻结旧SHA且CRC正常，明确按旧官方快照使用；若原快照缺失，不混用新版行情，不关闭校验。此工具不证明CSV内容改变。

工具提交锁定，只有I/O适配，不改原金融代码、账本或指纹检查。新运行时无法访问旧/content，请核对路径。

In [ ]:
# 将本格复制到原ER Notebook，等原⑤结束后运行。新运行时看不到旧/content。
from pathlib import Path
import sys,os,json,hashlib,getpass,urllib.request,base64,subprocess,signal
REPO=Path('/content/er-v2-retests-code-6a26255')
RESULTS=Path('/content/er_v2_retests_results_6a26255')
DATA=Path('/content/btc_eth_market_cache')
DRIVE_RESULTS=Path('/content/drive/MyDrive/btc_eth_er_v2_retests/6a26255')
RESUME=False # 首次只检查；原⑤结束且检查显示冻结快照有效后改True再运行
assert (REPO/'outputs/er_v2_retests/pipeline.py').is_file() and RESULTS.is_dir(), '不是原ER运行时或路径不同：请在原Notebook中运行本格，并核对上方目录。'
try:
    from google.colab import userdata
    token=userdata.get('GITHUB_TOKEN')
except Exception: token=getpass.getpass('GitHub token (hidden): ')
assert token, '缺少私有仓库读取凭据'
TOOL_COMMIT='ec73e9c2a791d2106f06453e60ef8941fab432c5'
HASHES={'cache_doctor.py': '530ca2c32db2bc66442abdc241d8f1a78ad46622b57bbd12e05f37d8cfb7ee51', 'resume_er_cache.py': 'f5f86fc3e927fba0db6b604381c3872a3b5dbcfa3a1357d0ae26f0141c7cdd8a'}
TOOLS=Path('/content/er_cache_recovery_tools');TOOLS.mkdir(exist_ok=True)
try:
    for name,expected in HASHES.items():
        url=f'https://api.github.com/repos/HITGoose/btc-eth-backtest/contents/outputs/ma_direction_retests/{name}?ref={TOOL_COMMIT}'
        req=urllib.request.Request(url,headers={'Authorization':'Bearer '+token,'Accept':'application/vnd.github+json'})
        with urllib.request.urlopen(req,timeout=60)as response: payload=json.load(response)
        content=base64.b64decode(payload['content']);assert hashlib.sha256(content).hexdigest()==expected
        (TOOLS/name).write_bytes(content)
finally: del token
cmd=[sys.executable,'-u',str(TOOLS/'resume_er_cache.py'),'--repo-root',str(REPO),'--results-root',str(RESULTS),'--data-root',str(DATA)]
if DRIVE_RESULTS.is_dir():cmd+=['--drive-results',str(DRIVE_RESULTS)]
if RESUME:cmd+=['--resume']
process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
try:
    for line in process.stdout:print(line,end='',flush=True)
    assert process.wait()==0, '检查或恢复失败；保留原件，不关闭校验。见上方具体错误。'
except KeyboardInterrupt:
    try:os.killpg(process.pid,signal.SIGTERM)
    except ProcessLookupError:pass
    process.wait();raise
